# Red Cross Donor Briefing — AGOL Notebook

Run this inside `arc-nhq-gis.maps.arcgis.com` under **Content → Notebook → New**.  
`GIS('home')` auto-authenticates as the notebook owner (you) — no OAuth app needed.

**Items we'll read:**
| Purpose | Item ID |
|---|---|
| Donors $5K+ past 2 years | `394f3b4115b3401f93944e7d40feeace` |
| Smoke alarm installs (10 yr) | `824e8beb0581464daec1892b5fdd8352` |
| (unlabeled — likely DAT/calls) | `b09f21d9257f4571ab643d860a8988f3` |

**This notebook:**
1. Introspects each layer — title, type, field schema, feature count
2. Samples first 3 rows of each so we learn real field names
3. After that, we'll add: donor-county resolution, spatial-join to incidents, briefing generation

No donor data leaves this notebook until you explicitly export it.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, FeatureLayerCollection
import pandas as pd
import json

gis = GIS('home')
print('Authenticated as:', gis.users.me.username)
print('Org URL:', gis.url)

In [ ]:
ITEMS = {
    'donors_5k_2yr': '394f3b4115b3401f93944e7d40feeace',
    'smoke_alarms_10yr': '824e8beb0581464daec1892b5fdd8352',
    'third_item': 'b09f21d9257f4571ab643d860a8988f3',
}

items = {}
for name, iid in ITEMS.items():
    it = gis.content.get(iid)
    items[name] = it
    if it is None:
        print(f'{name}: NOT FOUND — check permissions/id'); continue
    print(f'{name}:')
    print(f'  title   : {it.title}')
    print(f'  type    : {it.type}')
    print(f'  owner   : {it.owner}')
    print(f'  access  : {it.access}')
    print(f'  url     : {it.url}')
    print(f'  tags    : {it.tags}')
    print()

## Schema inspection

For each item that resolves to a FeatureService / FeatureLayer, print:
- field list
- feature count (sample a `where=1=1 returnCountOnly`)
- first 3 sample rows

If any item is a Web Map instead of a layer, we'll note that and dig into its operational layers.

In [ ]:
def introspect(item, max_layers=3, sample_rows=3):
    if item is None: return
    print('=' * 60)
    print(f'{item.title}  ({item.type})')
    print('=' * 60)

    # Feature Service → iterate layers
    if item.type in ('Feature Service', 'Feature Layer', 'Feature Layer Collection'):
        try:
            flc = FeatureLayerCollection.fromitem(item)
            layers = flc.layers or []
            print(f'Contains {len(layers)} layer(s).')
            for i, layer in enumerate(layers[:max_layers]):
                print(f'\n-- Layer {i}: {layer.properties.name} --')
                try:
                    count = layer.query(where='1=1', return_count_only=True)
                    print(f'Row count: {count}')
                except Exception as e:
                    print(f'(count failed: {e})')
                fields = layer.properties.fields
                print(f'Fields ({len(fields)}):')
                for f in fields:
                    print(f'    - {f.name} ({f.type}) alias={f.alias}')
                try:
                    df = layer.query(where='1=1', result_record_count=sample_rows, return_geometry=False).sdf
                    print(f'\nFirst {sample_rows} rows:')
                    # limit columns for readability
                    cols = [c for c in df.columns if c.lower() not in ('shape',)]
                    print(df[cols].head(sample_rows).to_string())
                except Exception as e:
                    print(f'(sample failed: {e})')
        except Exception as e:
            print(f'Could not treat as FeatureLayerCollection: {e}')
    elif item.type == 'Web Map':
        try:
            from arcgis.mapping import WebMap
            wm = WebMap(item)
            print(f'Operational layers: {len(wm.layers)}')
            for lyr in wm.layers:
                print(f'  - {lyr.get("title")} | url={lyr.get("url")} | itemId={lyr.get("itemId")}')
        except Exception as e:
            print(f'WebMap parse failed: {e}')
    else:
        print(f'(unhandled type — inspect manually: item.url = {item.url})')
    print()

for name, it in items.items():
    print(f'\n### {name} ###')
    introspect(it)

## Next cells (to add once we see the schemas above)

1. **Donor geocode check** — does the donor layer have lat/lon, or just address? Geocode if needed (using the org's World Geocoder).
2. **County resolver** — join donors to FL county polygons (we already have the list in `src/geography/fl_counties.py`; bring it in here too).
3. **Incident proximity** — spatial join donors to smoke-alarm installs and DAT/call incidents. Apply radius rule.
4. **Fatigue filter** — track "last touched" per donor in a Touch History hosted table.
5. **Briefing generation** — per donor, compose the county/chapter/region narrative. Hit the public `api/brief?county=...` endpoint or replicate inline.
6. **Output** — write a "Pending Donor Briefings" hosted table viewable in Experience Builder.

After you run the above and paste me the schema output, I'll write cells 1–6 against the real field names.